# Business Intelligence Showcase: SuperMart Case Study

> **Goal**: Play the role of a Senior Data Analyst. Carry out an end-to-end analysis to identify business problems and provide actionable recommendations to the CEO.

---

###  Phase 1: Setup & Data Connection

First, we connect to our database and ensure the data is loaded correctly.

In [ ]:
import pandas as pd
import sqlite3
import os
import matplotlib.pyplot as plt
import seaborn as sns

# Set style for visualizations
sns.set_theme(style="whitegrid")

# Connection setup
DATABASE = "../../retail_analytics.db"
if not os.path.exists(DATABASE):
    DATABASE = "retail_analytics.db"

conn = sqlite3.connect(DATABASE)
print("Connected to Database Successfully!")

###  Phase 2: SQL Extraction (City Performance)

**What**: Pulling store-level performance metrics.
**Why**: To identify which cities are driving revenue and which have low average spend.

In [ ]:
query_city = """
SELECT 
    s.location, 
    COUNT(t.txn_id) as total_orders,
    SUM(t.amount) as total_revenue,
    AVG(t.amount) as avg_order_value
FROM transactions t
JOIN stores s ON t.store = s.store
GROUP BY s.location
ORDER BY total_revenue DESC
"""

city_performance = pd.read_sql(query_city, conn)
city_performance

###  Phase 3: Sales Trend Analysis (Moving Averages)

**What**: Smoothing daily sales data.
**Why**: To remove "noise" and see the underlying growth trend.

In [ ]:
df = pd.read_sql("SELECT * FROM transactions", conn)
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date')

# Group by date for line chart
df_daily = df.groupby('date')['amount'].sum().reset_index()
df_daily['7day_moving_avg'] = df_daily['amount'].rolling(window=7).mean()

plt.figure(figsize=(12, 6))
sns.lineplot(data=df_daily, x='date', y='amount', label='Daily Sales', alpha=0.3)
sns.lineplot(data=df_daily, x='date', y='7day_moving_avg', label='7-Day Trend', color='red', linewidth=2)
plt.title('SuperMart Sales Trend: Signal vs. Noise')
plt.show()

###  Phase 4: Customer Health Check

**Who**: Premium members (member=1) who haven't shopped in the last 30 days.
**Why**: Retention is cheaper than acquisition. We need to win these people back.

In [ ]:
latest_date = df['date'].max()
thirty_days_ago = latest_date - pd.Timedelta(days=30)

dormant_champions = df[
    (df['member'] == 1) & 
    (df['date'] < thirty_days_ago)
].groupby('txn_id').first()

print(f"Latest Date in Data: {latest_date.date()}")
print(f"Number of Dormant Champions to target: {len(dormant_champions)}")

###  Phase 5: Executive Recommendations

Using the Minto Pyramid style (Conclusion first):

1. **Recommendation**: Launch 'VIP Re-engagement' campaign in Bangalore.
   *   *Evidence*: We have a high number of dormant premium members in that region.
2. **Recommendation**: Optimize supply chain for low-AOV cities.
   *   *Evidence*: Profit margins are thin in cities like Chennai due to low average spend per order.
3. **Recommendation**: Prepare for the mid-month dip.
   *   *Evidence*: The 7-day moving average consistently drops between the 15th-20th of every month.